In [1]:
import pandas as pd
import lightgbm as lgb

# 读取完整训练集、测试集和提交样例
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
submission = pd.read_csv("../data/sample_submission.csv")

# 检查数据规模
print("训练集:", train.shape)
print("测试集:", test.shape)
print("提交样例:", submission.shape)

训练集: (594194, 21)
测试集: (254655, 20)
提交样例: (254655, 2)


In [2]:
# 提取特征与目标变量
X = train.drop(columns=["id", "Churn"])
y = (train["Churn"] == "Yes").astype(int)

X_test = test[X.columns].copy()

# 与之前的 LightGBM 保持一致：使用原生类别特征
categorical_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

for col in categorical_cols:
    X[col] = X[col].astype("category")

    # 确保测试集使用与训练集一致的类别定义
    X_test[col] = pd.Categorical(
        X_test[col],
        categories=X[col].cat.categories
    )

print("训练特征:", X.shape)
print("测试特征:", X_test.shape)
print("类别变量数量:", len(categorical_cols))
print("训练与测试字段一致:", X.columns.equals(X_test.columns))

训练特征: (594194, 19)
测试特征: (254655, 19)
类别变量数量: 15
训练与测试字段一致: True


In [3]:
from lightgbm import LGBMClassifier

# 使用 03_lightgbm_model.ipynb 中已经验证的参数
# 将下面的参数替换为之前模型的实际参数
final_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbosity=-1
)

# 使用全部有标签数据训练
final_model.fit(
    X,
    y,
    categorical_feature=categorical_cols
)

print("最终 LightGBM 模型训练完成！")

最终 LightGBM 模型训练完成！


In [4]:
from lightgbm import LGBMClassifier

# 使用之前验证过的 LightGBM 参数
final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

# 使用完整训练集重新训练
final_model.fit(
    X,
    y,
    categorical_feature=categorical_cols
)

print("最终 LightGBM 模型训练完成！")
print("实际训练树数量:", final_model.n_estimators_)

最终 LightGBM 模型训练完成！
实际训练树数量: 300


In [5]:
# 预测测试集客户流失概率
test_proba = final_model.predict_proba(X_test)[:, 1]

# 检查预测结果
print("预测数量:", len(test_proba))
print("最小预测概率:", test_proba.min())
print("最大预测概率:", test_proba.max())

预测数量: 254655
最小预测概率: 0.0004193636819920882
最大预测概率: 0.9783792269927308


In [6]:
# 按 Kaggle 提交样例构造结果
submission["Churn"] = test_proba

# 保存到 reports 目录
submission.to_csv(
    "../reports/submission.csv",
    index=False
)

print("Submission 文件生成成功！")
print(submission.shape)

submission.head()

Submission 文件生成成功！
(254655, 2)


,id,Churn
0,594194,0.053144
1,594195,0.001021
2,594196,0.109454
3,594197,0.003490
4,594198,0.510573


In [7]:
# 检查提交文件
assert submission.shape == (254655, 2)
assert submission.columns.tolist() == ["id", "Churn"]
assert submission["id"].equals(test["id"])
assert submission["Churn"].notna().all()
assert submission["Churn"].between(0, 1).all()

print("所有提交格式检查通过！")

所有提交格式检查通过！
